# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-1002/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb scikit-learn
import duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")
print("Connected")

Connected


In [3]:
files = con.sql("SELECT file FROM glob('hf://datasets/FlyRank/internship-warehouse/**') LIMIT 100").df()
print(files.to_string())

                                                                                                      file
0                                                hf://datasets/FlyRank/internship-warehouse/.gitattributes
1                                                     hf://datasets/FlyRank/internship-warehouse/README.md
2                                           hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3                                           hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
8   hf://datasets/FlyRank/internship-

In [5]:
print(con.sql("SELECT content FROM read_text('hf://datasets/FlyRank/internship-warehouse/README.md')").fetchone()[0][:6000])

---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_query_90d
  data_files: fac

In [6]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.sql(f"DESCRIBE SELECT * FROM '{TABLE}'").show(max_rows=100)

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gs

In [7]:
con.sql(f"SELECT * FROM '{TABLE}' LIMIT 5").show(max_width=250)
con.sql("DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'").show(max_rows=100)

┌─────────────┬──────────────────────┬──────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬───┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │    client_hash_id    │   content_hash_id    │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ … │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_events │  month  │
│    date     │       varchar        │       varchar        │    boolean     │    boolean     │      boolean       │      boolean       │   │     int64     │   int64   │   int64    │   int64   │  int64  │  int64   │     int64     │ varchar │
├─────────────┼──────────────────────┼──────────────────────┼────────────────┼────────────────┼────────────────────┼────────────────────┼───┼───────────────┼───────────┼────────────┼───────────┼─────────┼──────────┼───────────────┼─────────┤
│ 2026-03-01  │ client_73cda7b4e

In [8]:
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

con.execute(f"""
CREATE OR REPLACE TABLE mar AS
SELECT report_date, client_hash_id, content_hash_id,
       gsc_data_available, gsc_impressions, gsc_clicks, gsc_avg_position
FROM '{FACT}'
""")
con.execute(f"""
CREATE OR REPLACE TABLE dc AS
SELECT client_hash_id, content_hash_id,
       MAX(content_updated_date) AS content_updated_date,
       MAX(word_count) AS word_count
FROM '{DIM}' GROUP BY 1, 2
""")
print("loaded", con.sql("SELECT COUNT(*) FROM mar").fetchone()[0], "rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

loaded 9841378 rows


## 1. Unit of analysis + time window
**One row = one content item of one client on one report date** (the daily fact table).
**Time window:** month = 2026-03, a mid-panel month used to build and test the logic.
Within it, days 1-15 are the "known" period (features) and days 16-31 are the
"outcome" period (label). June 2026, the final month, is kept sealed as a test month.

In [9]:
con.sql("""
SELECT COUNT(*) AS n_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM mar)) AS n_distinct_keys
FROM mar
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────────────┐
│ n_rows  │ n_distinct_keys │
│  int64  │      int64      │
├─────────┼─────────────────┤
│ 9841378 │         9841378 │
└─────────┴─────────────────┘



## 2. Fields: feature / label / context / excluded

**Features (5, all from days 1-15 of March or static page facts):**
- impr_h1: total GSC impressions in days 1-15, known before the decision day
- avg_pos_h1: average search position in days 1-15, measured in that same period
- active_days_h1: days with impressions in days 1-15, known from the same period
- days_since_update: days from last content update to 15 March, known from the page history
- word_count: a property of the page itself

**Label (proxy):** declining = 1 when the daily impression rate in days 16-31 is below
80% of the rate in days 1-15. "Needs a refresh" is not recorded, so this is a proxy.

**Context (not used as features):** client_hash_id, content_hash_id, report_date.

**Excluded, and why:** all impressions from days 16-31, because the label is built
from them (leakage). Also GA4, AI-referral and session columns, because many rows
have no GA4 data. Also dim_clients and any identifying fields.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
con.sql("""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
FROM mar
""").show()

┌─────────┬────────────┬────────────┬───────────┬───────────┐
│ n_rows  │ first_day  │  last_day  │ n_clients │ n_content │
│  int64  │    date    │    date    │   int64   │   int64   │
├─────────┼────────────┼────────────┼───────────┼───────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │        55 │    331437 │
└─────────┴────────────┴────────────┴───────────┴───────────┘



In [11]:
con.sql("""
SELECT COUNT(*) AS all_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS available_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS not_available,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS null_flag
FROM mar
""").show()

┌──────────┬────────────────┬───────────────┬───────────┐
│ all_rows │ available_rows │ not_available │ null_flag │
│  int64   │     int64      │     int64     │   int64   │
├──────────┼────────────────┼───────────────┼───────────┤
│  9841378 │        3611061 │       6230317 │         0 │
└──────────┴────────────────┴───────────────┴───────────┘



In [12]:
con.execute("""
CREATE OR REPLACE TABLE frame AS
SELECT m.client_hash_id, m.content_hash_id,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions END)  AS impr_h1,
  AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS avg_pos_h1,
  COUNT(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN 1 END) AS active_days_h1,
  COALESCE(SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions END), 0) AS impr_h2
FROM mar m
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""")

df = con.sql("""
SELECT f.*,
  CASE WHEN d.content_updated_date <= DATE '2026-03-15'
       THEN DATE '2026-03-15' - d.content_updated_date END AS days_since_update,
  d.word_count
FROM frame f LEFT JOIN dc d USING (client_hash_id, content_hash_id)
WHERE f.impr_h1 >= 50
""").df()

df["label"] = ((df["impr_h2"] / 16) < 0.8 * (df["impr_h1"] / 15)).astype(int)
df = df.sample(min(len(df), 100_000), random_state=42).reset_index(drop=True)
FEATURES = ["impr_h1", "avg_pos_h1", "active_days_h1", "days_since_update", "word_count"]
print(df.shape, "| share label=1:", round(df["label"].mean(), 3))
df[FEATURES + ["label"]].head()

(92548, 9) | share label=1: 0.327


,impr_h1,avg_pos_h1,active_days_h1,days_since_update,word_count,label
0,3251.0,37.183161,15,<NA>,3532,0
1,67.0,6.479594,14,<NA>,<NA>,1
2,435.0,1.753987,15,<NA>,<NA>,0
3,2338.0,2.387227,15,18,<NA>,0
4,90.0,23.582568,15,<NA>,<NA>,1


In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    tr, te = next(GroupShuffleSplit(test_size=0.3, random_state=42).split(X, df["label"], df["client_hash_id"]))
    m = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X.iloc[tr], df["label"].iloc[tr])
    return roc_auc_score(df["label"].iloc[te], m.predict_proba(X.iloc[te])[:, 1])

print("Honest AUC:        ", round(quick_score(FEATURES), 3))
df["leak"] = df["impr_h2"]                       # label-derived, added on purpose
print("With leak AUC:     ", round(quick_score(FEATURES + ["leak"]), 3), " <- jumps toward perfect")
df = df.drop(columns=["leak"])                   # deleted
print("Back to honest AUC:", round(quick_score(FEATURES), 3))

Honest AUC:         0.542
With leak AUC:      0.717  <- jumps toward perfect
Back to honest AUC: 0.542


## 4. Data limits
- The panel is unbalanced: clients have different history depths (dim_clients records
  when GSC and GA4 data start), so early rows may have only Search Console data.
- Many rows have no GA4 data, which is why I use only GSC fields.
- My label compares days 16-31 with days 1-15 of ONE month. It is a short window,
  a proxy for "declining", and sensitive to day-to-day noise.
- A page with no rows in days 16-31 counts as zero impressions, which may mean
  missing data rather than a real drop.
- Clients differ a lot in size, so results may not hold for every site.
- It is observational and directional. It supports a decision and proves no cause.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.